# Lab 7.11 &mdash; Assemble the Email Agent

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 4 &middot; Module 7 &mdash; Task Automation with AI Agents**

### What you'll do
- Assemble gather tools + model + prompt into an AgentExecutor
- Withhold send_email so the agent cannot auto-send
- Return a grounded draft flagged needs_approval, with its trace

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** the graded steps are **offline &amp; deterministic** (pure Python stdlib); the agent-assembly labs reuse the **LangChain-shaped shim** from Module 6. Advanced labs end with an **optional** cell that runs the **real** library. You are building the **email-drafting agent** &mdash; the client's Lab 4.1.

**Reference:** [Module 7 slides &mdash; The email-drafting agent, end to end](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 7 labs](./index.html)

In [ ]:
# Setup -- run me first
import os
WORK = "/tmp/biaa-lab-07-11"
os.makedirs(WORK, exist_ok=True)
print("Working dir:", WORK)

## Concept
Now assemble the email agent from Module 6's pieces (deck slides 14&ndash;16): `@tool` gather tools,
`create_react_agent`, an `AgentExecutor` with a **`max_iterations`** cap. The scripted model gathers
(`lookup_order`, `get_template`) then drafts a **Final Answer**. The key design choice: the tools
list is **gather-only** &mdash; `send_email` is **not** given &mdash; and the run's output is wrapped as
a **`needs_approval`** draft. The agent did the tedious 90%; the human keeps the send.

In [ ]:
# --- LangChain-SHAPED shim: a tool has .name, .description (from the docstring), .args, .invoke() ---
import inspect
class Tool:
    def __init__(self, fn, name=None, description=None):
        self.fn = fn
        self.name = name or fn.__name__
        self.description = (description or inspect.getdoc(fn) or "").strip()
        self.args = list(inspect.signature(fn).parameters)
    def invoke(self, value):
        return self.fn(**value) if isinstance(value, dict) else self.fn(value)
    def __repr__(self):
        return "Tool(name=%r)" % self.name
def tool(fn):
    """The @tool decorator -- wrap a plain function into a Tool (mirrors langchain_core.tools.tool)."""
    return Tool(fn)

class AIMessage:
    def __init__(self, content): self.content = content
class FakeChatModel:
    """Deterministic stand-in for ChatOllama / ChatGroq: replays a scripted list of replies.
    Real code: from langchain_ollama import ChatOllama; model = ChatOllama(model="llama3.2:1b").
    Like the real thing, .invoke(prompt) returns a message whose text is in .content."""
    def __init__(self, script): self.script = list(script); self.i = 0
    def invoke(self, prompt):
        reply = self.script[min(self.i, len(self.script) - 1)]; self.i += 1
        return AIMessage(reply)

class PromptTemplate:
    """Mirrors LangChain: PromptTemplate.from_template(...).format(input=..., ...)."""
    def __init__(self, template): self.template = template
    @classmethod
    def from_template(cls, template): return cls(template)
    def format(self, **kw):
        s = self.template
        for k, v in kw.items():
            s = s.replace("{" + k + "}", str(v))
        return s

def create_react_agent(model, tools, prompt):
    """Bind model + tools + prompt into a ReAct agent (mirrors langchain.agents.create_react_agent)."""
    return {"model": model, "tools": {t.name: t for t in tools}, "prompt": prompt}
def parse_react(text):
    """Turn the model's ReAct text into ('final', answer) or ('action', name, input)."""
    action = inp = None
    for line in text.splitlines():
        s = line.strip()
        if s.startswith("Final Answer:"): return ("final", s.split(":", 1)[1].strip())
        if s.startswith("Action Input:"): inp = s.split(":", 1)[1].strip()
        elif s.startswith("Action:"):      action = s.split(":", 1)[1].strip()
    return ("action", action, inp)
class AgentExecutor:
    """Runs the loop: ask model -> parse -> run tool -> observe -> repeat, capped by max_iterations
    (mirrors langchain.agents.AgentExecutor). verbose=True prints the ReAct trace."""
    def __init__(self, agent, tools=None, verbose=False, max_iterations=6):
        self.agent = agent
        self.tools = agent["tools"] if tools is None else {t.name: t for t in tools}
        self.model = agent["model"]; self.prompt = agent["prompt"]
        self.verbose = verbose; self.max_iterations = max_iterations
    def invoke(self, inputs):
        scratch, steps = "", []
        for _ in range(self.max_iterations):
            text = self.model.invoke(self.prompt.format(input=inputs["input"], scratchpad=scratch)).content
            if self.verbose: print(text)
            parsed = parse_react(text)
            if parsed[0] == "final":
                return {"output": parsed[1], "intermediate_steps": steps}
            name, arg = parsed[1], parsed[2]
            obs = self.tools[name].invoke(arg) if name in self.tools else ("unknown tool: %s" % name)
            if self.verbose: print("Observation:", obs)
            steps.append((name, arg, obs)); scratch += text + "\nObservation: " + str(obs) + "\n"
        return {"output": None, "intermediate_steps": steps}

# The email agent's context sources: a small orders DB and a set of reply templates.
ORDERS = {
    "4471": {"id": "4471", "name": "Priya", "status": "shipped",    "eta": "Friday",    "carrier": "BlueDart"},
    "5090": {"id": "5090", "name": "Sam",   "status": "processing", "eta": "next week", "carrier": "-"},
}
TEMPLATES = {
    "delivery_delay": "Hi {name}, your order {id} has {status} and is due {eta}. Thanks for your patience.",
    "refund":         "Hi {name}, we've started the refund for order {id}. It will reflect in 5-7 days.",
}

@tool
def lookup_order(order_id):
    """Look up an order's status, ETA and carrier by id."""
    return ORDERS.get(order_id, {"status": "unknown"})
@tool
def get_template(kind):
    """Fetch a reply template by kind."""
    return TEMPLATES.get(kind, "")
print("gather tools ready:", lookup_order.name, "&", get_template.name)

## Your Turn
Complete `make_email_agent` (gather-only tools + the step cap) and `handle_email` (wrap the draft as
needs_approval).

In [ ]:
SCRIPT = [
    "Thought: get the order status first.\nAction: lookup_order\nAction Input: 4471",
    "Thought: fetch the reply template.\nAction: get_template\nAction Input: delivery_delay",
    "Thought: I can draft the reply now.\nFinal Answer: Hi Priya, your order 4471 has shipped and is due Friday.",
]

def make_email_agent(max_iterations=6):
    model  = FakeChatModel(SCRIPT)
    prompt = PromptTemplate.from_template("Email: {input}\n{scratchpad}")
    tools  = ___   # TODO: gather-only -- lookup_order & get_template, NO send_email
    agent  = create_react_agent(model, tools, prompt)
    return AgentExecutor(agent, max_iterations=___)   # TODO: the step cap

def handle_email(email):
    result = make_email_agent().invoke({"input": email})
    # never auto-send: return a DRAFT flagged for human approval
    return {"draft": result["output"], "status": ___,   # TODO: the needs-approval flag
            "tools_used": [s[0] for s in result["intermediate_steps"]]}

try:
    out = handle_email("Where is my order 4471? It's late.")
    print('tools used:', out['tools_used'])
    print('status    :', out['status'])
    print('draft     :', out['draft'])
    print('has send tool?', 'send_email' in [t.name for t in make_email_agent().tools.values()])
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("the agent gathers order THEN template", lambda: handle_email("x")["tools_used"] == ["lookup_order", "get_template"])
expect_true("the draft is grounded in the order id", lambda: "4471" in handle_email("x")["draft"])
expect_true("the draft is grounded in the real ETA", lambda: "Friday" in handle_email("x")["draft"])
expect_true("the output is a needs_approval draft, never sent", lambda: handle_email("x")["status"] == "needs_approval")
expect_true("the agent holds NO send tool", lambda: "send_email" not in [t.name for t in make_email_agent().tools.values()])

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

## Optional &mdash; run this against the REAL LangChain (not graded)
Swap the scripted model for a REAL LangChain agent (Ollama llama3.2:1b, or Groq) -- gather-only, drafting. Safe to skip &mdash; it needs `pip install langchain langchain-ollama` (then
`ollama run llama3.2:1b`) or `langchain-groq` with a `GROQ_API_KEY`. If a package, model or key is
missing the cell prints a friendly note and moves on.
**The graded steps above are offline &amp; deterministic, so the lab always verifies without a model.**

In [ ]:
try:
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model="llama3.2:1b")
    draft = llm.invoke("Draft a one-line reply: order 4471 shipped, due Friday. Do NOT send it.").content
    print("REAL draft:", draft)
    print("In production: bind lookup_order/get_template with create_react_agent + AgentExecutor,")
    print("and simply DON'T include a send tool -- the agent can gather & draft but cannot send.")
except Exception as e:
    print("No local LLM available -- skipping (pip install langchain langchain-ollama + `ollama run llama3.2:1b`,")
    print("or langchain-groq with GROQ_API_KEY):", type(e).__name__)
    print("The offline agent above already produced a grounded needs_approval draft.")

---
### Key takeaway
Same executor as Module 6, pointed at a real job -- and the guardrail is what's MISSING from the tools list. The agent gathers and drafts; it cannot send. Next: run it over a whole suite.

[&#8592; All Module 7 labs](./index.html) &nbsp;&middot;&nbsp; [Module 7 slides](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>